# Run felsenstein on simulated and recovered trees

Some steps to take

- Run felsenstein once per tree. Its possible multiple SNPs share the same tree.
we can store which tree contain a snp of interest
- chnage so that tree length is cached, we can reuse so that we dont have to recqlculate

In [1]:
import tskit
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, SVG
from scipy.linalg import expm
from collections import defaultdict
import time

from python_code.felsenstein_codes import (
felsenstein_backwards_general,
felsenstein_backwards_two_state,
snps_by_tree_index,
tip_states_at)

from python_code.data_utils import (
    haplotype_columns,
    mutation_rate_from_provenance,
    mutation_branches,
    clade_membership,
    correlate_clades,
    clade_correlations)


In [ ]:
# Load the tree sequences

filename = "chr12_HapMapII_GRCh37_CEU50_seed1"
percent = "10"
num_bins = "20"

# reference ts from stdpopsim
ts_std = tskit.load(f"../../stdpopsim_simulations/simulations/{filename}.trees")
# masked ts from Relate
ts_masked = tskit.load(f"../../Relate_sims/masked_{filename}_{percent}.00percent_{num_bins}bins_seed42.trees")

# load in hidden SNP indices
hidden_snps_df = pd.read_csv(f"../../Relate_sims/hidden_snps_key_{filename}_{percent}.00percent_{num_bins}bins_seed42.csv")


In [5]:
hidden_snps_df.head(30)

,Unnamed: 0,Chromosome number,SNP ID,SNP position,Ancestral allele,Alternative allele,5,6,7,8,...,997,998,999,1000,1001,1002,1003,1004,ALT_count,ALT_count_numbered_bin
0,478742,12,479355,130919180,A,T,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,330678,12,331094,91013145,C,T,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2,85786,12,85891,24190668,A,C,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
3,367077,12,367544,100836424,G,T,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,97645,12,97766,27350327,A,C,0,0,0,0,...,0,0,0,0,0,0,0,0,10,0
5,297211,12,297590,81694959,T,G,0,0,0,0,...,0,0,0,0,0,0,0,0,4,0
6,183904,12,184137,50685621,T,A,0,0,0,0,...,0,0,1,0,0,0,0,0,119,2
7,374298,12,374768,102790020,C,T,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
8,347498,12,347933,95676605,C,A,0,0,0,0,...,0,0,0,0,0,0,0,0,13,0
9,13398,12,13416,3843630,C,A,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


## single tree example

In [6]:
pos = 50685621	

In [7]:
# get the trees associated to the snps so that the same tree doesnt run twice (this is for all trees btw)
snp_to_tree_std = snps_by_tree_index(ts_std, hidden_snps_df["SNP position"])
snp_to_tree_masked= snps_by_tree_index(ts_masked, hidden_snps_df["SNP position"])

skipped 2 SNPs outside the sequence or on empty trees


In [8]:
# what are the true tip states for felsenstein at this position
tips = tip_states_at(hidden_snps_df, pos)

In [9]:
# get only usable snps
hap_cols = haplotype_columns(hidden_snps_df)
biallelic = hidden_snps_df[hidden_snps_df[hap_cols].max(axis=1) <= 1]["SNP position"]

in_std = {p for ps in snp_to_tree_std.values() for p in ps}
in_masked = {p for ps in snp_to_tree_masked.values() for p in ps}

usable = [p for p in biallelic if p in in_std and p in in_masked]
print(len(hidden_snps_df), "hidden SNPs,", len(usable), "usable")

48753 hidden SNPs, 48751 usable


In [10]:
# any zero length branches ?
lengths = ts_masked.nodes_time[ts_masked.edges_parent] - ts_masked.nodes_time[ts_masked.edges_child]
print("zero-length branches:", (lengths <= 0).sum())

zero-length branches: 0


In [11]:
ts_std

In [12]:
# relative mutation rates of each
rate_std = mutation_rate_from_provenance(ts_std)[0]
rate_relate = 1.25e-8


In [13]:
carriers = {s for s, g in tips.items() if g == 1}
print(f"SNP at {pos:,.0f}, {len(carriers)} carriers")
if len(carriers) <= 20:
    print("  carriers:", sorted(carriers))
print()

for name, ts, rate in (("stdpopsim", ts_std, rate_std), ("relate", ts_masked, rate_relate)):
    tree = ts.at(pos)
    log_marginal, states, log_joint, num_mutations = felsenstein_backwards_general(
        tree, tips, bases=(0, 1), mutation_rate=rate, root_state=0)

    # clade correlation for every branch, with your functions
    node_ids, sample_ids, membership = clade_membership(tree, include_root=False)
    carrier = np.array([tips[int(s)] for s in sample_ids])
    corr = correlate_clades(membership, carrier)
    phi = dict(zip(node_ids.tolist(), corr))
    best = int(node_ids[np.nanargmax(corr)])

    # marginal: all histories added up; joint: the single best history (both already logs)
    best_share = np.exp(log_joint - log_marginal)   # what share of the sum is the best

    print(f"{name}:")
    print(f"  marginal (all histories):  log {log_marginal:.3f}")
    print(f"  joint (best history):      log {log_joint:.3f}")
    print(f"  best history's share:      {best_share:.1%} of the marginal")
    branches = mutation_branches(tree, states)

    # label each change relative to the root's (ancestral) state
    ancestral = states[tree.root]
    kinds = {}
    for child, parent in branches:
        if states[parent] == ancestral:
            kinds[child] = "new mutation"        # ancestral -> derived
        elif states[child] == ancestral:
            kinds[child] = "back mutation"       # derived -> back to ancestral
        else:
            kinds[child] = "derived -> derived"  # only possible with 3+ states
    n_new = sum(k == "new mutation" for k in kinds.values())
    n_back = sum(k == "back mutation" for k in kinds.values())
    print(f"  best history: {n_new + n_back} total mutation(s)")
    print(f"  {n_new} new mutation(s), {n_back} back mutation(s)")

    for child, parent in branches[:10]:                    # show at most 10, oldest first
        below = set(tree.samples(child))
        n_car, n_non = len(below & carriers), len(below - carriers)
        print(f"    {kinds[child]} ({states[parent]} -> {states[child]}) on branch {parent} -> {child}, "
              f"time {ts.nodes_time[child]:,.0f} - {ts.nodes_time[parent]:,.0f} generations, "
              f"clade correlation {phi[child]:.3f}")
        if kinds[child] == "new mutation":
            print(f"      gives the derived allele to the {len(below)} samples below: "
                  f"{n_car} carriers, {n_non} non-carriers")
        elif kinds[child] == "back mutation":
            print(f"      reverts the {len(below)} samples below to the ancestral allele: "
                  f"{n_non} non-carriers, {n_car} carriers")
        if len(below) <= 20:
            print(f"      samples below: {sorted(below)}")
    if len(branches) > 10:
        print(f"    ... and {len(branches) - 10} more")
    print(f"  best clade correlation:    {phi[best]:.3f} (branch above node {best}, "
          f"{tree.num_samples(best)} samples below)")
    print()

print("truth (from stdpopsim):")
tree = ts_std.at(pos)
print(f"  true number of mutations: {len(ts_std.site(position=pos).mutations)}")
for m in ts_std.site(position=pos).mutations:
    print(f"  above node {m.node}, {tree.num_samples(m.node)} samples below, "
          f"at {m.time:,.0f} generations")

SNP at 50,685,621, 119 carriers

stdpopsim:
  marginal (all histories):  log -9.436
  joint (best history):      log -9.436
  best history's share:      100.0% of the marginal
  best history: 1 total mutation(s)
  1 new mutation(s), 0 back mutation(s)
    new mutation (0 -> 1) on branch 199606 -> 160797, time 851 - 7,057 generations, clade correlation 1.000
      gives the derived allele to the 119 samples below: 119 carriers, 0 non-carriers
  best clade correlation:    1.000 (branch above node 160797, 119 samples below)

relate:
  marginal (all histories):  log -9.886
  joint (best history):      log -9.886
  best history's share:      100.0% of the marginal
  best history: 1 total mutation(s)
  1 new mutation(s), 0 back mutation(s)
    new mutation (0 -> 1) on branch 6259615 -> 6259614, time 2,638 - 6,720 generations, clade correlation 1.000
      gives the derived allele to the 119 samples below: 119 carriers, 0 non-carriers
  best clade correlation:    1.000 (branch above node 6259

# All valid sites run

In [14]:
# ---------------------------------------------------------------
# 1. settings
# ---------------------------------------------------------------
rate_std = mutation_rate_from_provenance(ts_std)[0]    # true rate from stdpopsim
rate_relate = 1.25e-8                                   # the -m you gave Relate
root_state = 0                                          # root fixed to the true ancestral allele

# ---------------------------------------------------------------
# 2. tip states for every hidden SNP, read from the dataframe once
# ---------------------------------------------------------------
hap_cols = haplotype_columns(hidden_snps_df)
assert len(hap_cols) == ts_std.num_samples == ts_masked.num_samples

H = hidden_snps_df[hap_cols].to_numpy().astype(int)            # one row per SNP, one column per haplotype
snp_positions = hidden_snps_df["SNP position"].to_numpy()
row_of = {p: i for i, p in enumerate(snp_positions.tolist())}  # position -> row in H

# ---------------------------------------------------------------
# 3. which SNPs to run: two alleles only, and a tree in both tree sequences
# ---------------------------------------------------------------
snp_to_tree_std = snps_by_tree_index(ts_std, snp_positions)
snp_to_tree_masked = snps_by_tree_index(ts_masked, snp_positions)

biallelic = {p for p, row in zip(snp_positions.tolist(), H) if row.max() <= 1 and 0 < row.sum() < len(row)}
in_std = {p for ps in snp_to_tree_std.values() for p in ps}
in_masked = {p for ps in snp_to_tree_masked.values() for p in ps}
usable = biallelic & in_std & in_masked
# usable = set(sorted(usable)[:2000])                  # uncomment to try a small batch first
print(f"{len(snp_positions):,} hidden SNPs, {len(usable):,} usable")



skipped 2 SNPs outside the sequence or on empty trees
48,753 hidden SNPs, 48,751 usable


In [15]:


# ---------------------------------------------------------------
# 4. run Felsenstein on every usable SNP, tree by tree, for both tree sequences
# ---------------------------------------------------------------
results = {pos: {} for pos in usable}      # pos -> {"std_...": ..., "relate_...": ...}

for name, ts, rate, snp_to_tree in (("std", ts_std, rate_std, snp_to_tree_std),
                                    ("relate", ts_masked, rate_relate, snp_to_tree_masked)):
    start, done = time.time(), 0
    tree = ts.first()
    for tree_index in sorted(snp_to_tree):
        positions_here = [p for p in snp_to_tree[tree_index] if p in usable]
        if not positions_here:
            continue
        tree.seek_index(tree_index)                        # move to this tree (once for all its SNPs)
        samples_below = {u: tree.num_samples(u) for u in tree.nodes()}
        cache = {}                                         # same tip pattern on this tree -> same result

        for pos in positions_here:
            haps = H[row_of[pos]]
            key = haps.tobytes()
            if key not in cache:
                tips = dict(enumerate(haps.tolist()))

                # Felsenstein: marginal and joint, both as logs
                log_marginal, states, log_joint, num_mutations = felsenstein_backwards_two_state(
                    tree, tips, bases=(0, 1), mutation_rate=rate, root_state=root_state)

                # clade correlation of every branch (utils: clade_correlations)
                node_ids, corr = clade_correlations(tree, haps, include_root=False)
                phi = dict(zip(node_ids.tolist(), corr))
                best_node = int(node_ids[np.nanargmax(corr)])

                # label the changes in the best history
                ancestral = states[tree.root]
                changes = [u for u in states
                           if tree.parent(u) != -1 and states[u] != states[tree.parent(u)]]
                new = [u for u in changes if states[tree.parent(u)] == ancestral]
                back = [u for u in changes if states[u] == ancestral]

                record = {
                    "log_marginal": log_marginal,
                    "log_joint": log_joint,
                    "best_share": np.exp(log_joint - log_marginal),
                    "mutations": num_mutations,
                    "new_mutations": len(new),
                    "back_mutations": len(back),
                    "best_clade_corr": phi[best_node],
                    "best_clade_size": samples_below[best_node],
                }
                # details of the main new mutation: the one with the most samples below
                # (the only one, if there is just one; ties go to the lower node ID so the choice is fixed)
                if new:
                    u = max(new, key=lambda v: (samples_below[v], -v))
                    record.update({
                        "new_branch_low": ts.nodes_time[u],
                        "new_branch_high": ts.nodes_time[tree.parent(u)],
                        "new_samples_below": samples_below[u],
                        "new_noncarriers_below": samples_below[u] - sum(tips[s] for s in tree.samples(u)),
                        "new_clade_corr": phi[u],
                    })
                cache[key] = record

            results[pos].update({f"{name}_{k}": v for k, v in cache[key].items()})
            done += 1
            if done % 2000 == 0:
                print(f"{name}: {done:,} / {len(usable):,} SNPs, {time.time() - start:.0f}s")
    print(f"{name}: finished {done:,} SNPs in {time.time() - start:.0f}s")


std: 2,000 / 48,751 SNPs, 16s
std: 4,000 / 48,751 SNPs, 30s
std: 6,000 / 48,751 SNPs, 44s
std: 8,000 / 48,751 SNPs, 59s
std: 10,000 / 48,751 SNPs, 73s
std: 12,000 / 48,751 SNPs, 87s
std: 14,000 / 48,751 SNPs, 97s
std: 16,000 / 48,751 SNPs, 110s
std: 18,000 / 48,751 SNPs, 124s
std: 20,000 / 48,751 SNPs, 138s
std: 22,000 / 48,751 SNPs, 152s
std: 24,000 / 48,751 SNPs, 166s
std: 26,000 / 48,751 SNPs, 180s
std: 28,000 / 48,751 SNPs, 195s
std: 30,000 / 48,751 SNPs, 209s
std: 32,000 / 48,751 SNPs, 223s
std: 34,000 / 48,751 SNPs, 237s
std: 36,000 / 48,751 SNPs, 251s
std: 38,000 / 48,751 SNPs, 265s
std: 40,000 / 48,751 SNPs, 280s
std: 42,000 / 48,751 SNPs, 294s
std: 44,000 / 48,751 SNPs, 308s
std: 46,000 / 48,751 SNPs, 322s
std: 48,000 / 48,751 SNPs, 336s
std: finished 48,751 SNPs in 342s
relate: 2,000 / 48,751 SNPs, 14s
relate: 4,000 / 48,751 SNPs, 27s
relate: 6,000 / 48,751 SNPs, 41s
relate: 8,000 / 48,751 SNPs, 54s
relate: 10,000 / 48,751 SNPs, 67s
relate: 12,000 / 48,751 SNPs, 81s
relate: 1

In [16]:
# ---------------------------------------------------------------
# 5. add the truth and put everything in one table
# ---------------------------------------------------------------
alt_bin = hidden_snps_df["ALT_count_numbered_bin"].to_numpy()   # your existing bins, same row order as H

rows = []
for pos in sorted(usable):
    site = ts_std.site(position=pos)
    rows.append({
        "position": pos,
        "ALT_count": int(H[row_of[pos]].sum()),
        "ALT_count_bin": alt_bin[row_of[pos]],
        "true_mutations": len(site.mutations),
        "true_mutation_time": site.mutations[0].time if len(site.mutations) == 1 else np.nan,
        **results[pos],
    })
results_df = pd.DataFrame(rows)

# is the true mutation time inside the branch each tree chose?
for name in ("std", "relate"):
    results_df[f"{name}_time_inside"] = (
        (results_df.true_mutation_time >= results_df[f"{name}_new_branch_low"])
        & (results_df.true_mutation_time <= results_df[f"{name}_new_branch_high"])
    )
results_df["dlog_marginal"] = results_df.relate_log_marginal - results_df.std_log_marginal
results_df["dlog_best"] = results_df.relate_log_joint - results_df.std_log_joint


In [17]:
results_df

,position,ALT_count,ALT_count_bin,true_mutations,true_mutation_time,std_log_marginal,std_log_joint,std_best_share,std_mutations,std_new_mutations,...,relate_best_clade_size,relate_new_branch_low,relate_new_branch_high,relate_new_samples_below,relate_new_noncarriers_below,relate_new_clade_corr,std_time_inside,relate_time_inside,dlog_marginal,dlog_best
0,149532,1,0,1,135.532352,-13.247778,-13.247778,1.000000,1,1,...,1,0.000000,7.337350,1,0,1.000000,True,False,-2.959862,-2.959862
1,150672,5,0,1,4386.115458,-8.599594,-8.599689,0.999905,1,1,...,5,64.340858,6390.902832,5,0,1.000000,True,True,-0.848426,-0.848417
2,150938,13,0,1,303.548132,-13.783608,-13.783611,0.999997,1,1,...,11,30.285500,37.130951,11,1,0.834296,True,False,-62.992723,-63.617943
3,154298,1,0,1,15.430524,-14.392934,-14.392934,1.000000,1,1,...,1,0.000000,5.560260,1,0,1.000000,True,False,-2.092039,-2.092039
4,155035,62,1,1,3242.205811,-10.581408,-10.581431,0.999977,1,1,...,67,739.551819,935.375671,67,20,0.710616,True,False,-234.879731,-235.931762
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48746,133257398,2,0,1,229.549824,-12.830935,-12.830936,0.999999,1,1,...,2,25.101681,78.389748,2,0,1.000000,True,False,-1.394406,-1.394405
48747,133260668,2,0,1,207.667106,-12.654485,-12.654486,0.999999,1,1,...,2,29.014351,109.677681,2,0,1.000000,True,False,-1.156284,-1.156283
48748,133262747,3,0,1,180.487032,-12.606161,-12.606163,0.999997,1,1,...,3,44.405170,202.389343,3,0,1.000000,True,True,-0.532396,-0.532396
48749,133267155,14,0,1,680.114992,-12.569630,-12.569643,0.999988,1,1,...,14,318.912415,1093.713501,14,0,1.000000,True,True,1.020831,1.020832


In [19]:
# save the results to a csv
# results_df.to_csv("felsenstein_2_state_results.csv", index=False)   # save, so you don't have to rerun
results_df.to_csv(f"felsenstein_{filename}_{percent}.00percent_{num_bins}bins_seed42.csv", index=False)
results_df.head(10)

,position,ALT_count,ALT_count_bin,true_mutations,true_mutation_time,std_log_marginal,std_log_joint,std_best_share,std_mutations,std_new_mutations,...,relate_best_clade_size,relate_new_branch_low,relate_new_branch_high,relate_new_samples_below,relate_new_noncarriers_below,relate_new_clade_corr,std_time_inside,relate_time_inside,dlog_marginal,dlog_best
0,149532,1,0,1,135.532352,-13.247778,-13.247778,1.000000,1,1,...,1,0.000000,7.337350,1,0,1.000000,True,False,-2.959862,-2.959862
1,150672,5,0,1,4386.115458,-8.599594,-8.599689,0.999905,1,1,...,5,64.340858,6390.902832,5,0,1.000000,True,True,-0.848426,-0.848417
2,150938,13,0,1,303.548132,-13.783608,-13.783611,0.999997,1,1,...,11,30.285500,37.130951,11,1,0.834296,True,False,-62.992723,-63.617943
3,154298,1,0,1,15.430524,-14.392934,-14.392934,1.000000,1,1,...,1,0.000000,5.560260,1,0,1.000000,True,False,-2.092039,-2.092039
4,155035,62,1,1,3242.205811,-10.581408,-10.581431,0.999977,1,1,...,67,739.551819,935.375671,67,20,0.710616,True,False,-234.879731,-235.931762
5,155054,3,0,1,40770.515108,-7.371807,-7.371807,1.000000,1,1,...,3,336.020538,6390.902832,3,0,1.000000,True,False,-2.120098,-2.120190
6,162334,1,0,1,191.514509,-11.844037,-11.844038,0.999999,1,1,...,1,0.000000,58.464180,1,0,1.000000,True,False,-2.288178,-2.288179
7,170830,1,0,1,174.519775,-12.183500,-12.183500,1.000000,1,1,...,1,0.000000,16.191210,1,0,1.000000,True,False,-3.232652,-3.232652
8,171435,2,0,1,3761.577430,-8.808423,-8.808456,0.999967,1,1,...,2,17.319910,153.660156,2,0,1.000000,True,False,-4.477041,-4.477012
9,172503,34,0,1,445.899176,-14.121649,-14.121650,0.999999,1,1,...,22,242.683990,362.142792,22,5,0.611361,True,False,-124.377962,-124.378015


# Summarize

In [20]:
multi = results_df[results_df.true_mutations > 1]
print(f"{len(multi)} of {len(results_df)} usable hidden SNPs have more than one true mutation")
print(results_df.true_mutations.value_counts().sort_index())

30 of 48751 usable hidden SNPs have more than one true mutation
true_mutations
1    48721
2       30
Name: count, dtype: int64


In [21]:
for pos in multi.position:
    site = ts_std.site(position=pos)
    tree = ts_std.at(pos)
    print(f"\nSNP at {pos:,.0f}: {len(site.mutations)} true mutations, ancestral state {site.ancestral_state}")
    for m in site.mutations:
        on_top = "" if m.parent == -1 else f" (on top of mutation {m.parent})"
        print(f"  mutation {m.id}: above node {m.node}, {tree.num_samples(m.node)} samples below, "
              f"at {m.time:,.0f} generations, -> {m.derived_state}{on_top}")
    row = results_df[results_df.position == pos].iloc[0]
    print(f"  Felsenstein on the true tree found {row.std_mutations} "
          f"({row.std_new_mutations} new, {row.std_back_mutations} back)")


SNP at 5,559,041: 2 true mutations, ancestral state G
  mutation 19736: above node 105628, 15 samples below, at 656 generations, -> T
  mutation 19737: above node 77290, 8 samples below, at 175 generations, -> T
  Felsenstein on the true tree found 2 (2 new, 0 back)

SNP at 10,338,161: 2 true mutations, ancestral state A
  mutation 36903: above node 169542, 126 samples below, at 13,556 generations, -> G
  mutation 36904: above node 156469, 178 samples below, at 1,168 generations, -> G
  Felsenstein on the true tree found 2 (2 new, 0 back)

SNP at 15,197,123: 2 true mutations, ancestral state C
  mutation 54359: above node 463, 1 samples below, at 100 generations, -> G
  mutation 54360: above node 495, 1 samples below, at 14 generations, -> G
  Felsenstein on the true tree found 2 (2 new, 0 back)

SNP at 30,928,295: 2 true mutations, ancestral state C
  mutation 111132: above node 185632, 120 samples below, at 4,184 generations, -> G
  mutation 111133: above node 47919, 3 samples below

In [22]:
per_site = np.bincount(ts_std.mutations_site, minlength=ts_std.num_sites)
print(np.unique(per_site, return_counts=True))   # (mutations per site, number of sites)

(array([1, 2, 3]), array([487235,    922,      2]))
